<a href="https://colab.research.google.com/github/achrafmontage/maliki-rag/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# نظام الفقه المالكي (RAG صارم)
شغّل الخلايا بالترتيب. اختر **Runtime → Change runtime type → T4 GPU** قبل البدء.
أضف مفتاح `ANTHROPIC_API_KEY` من أيقونة 🔑 (Secrets) في Colab.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# الفهرس الكبير يُحفظ على Drive كي لا يضيع بانتهاء الجلسة
os.environ['MALIKI_INDEX_DIR'] = '/content/drive/MyDrive/maliki-rag/index'
os.makedirs(os.environ['MALIKI_INDEX_DIR'], exist_ok=True)

## 1) استنساخ المستودع
غيّر `USER` إلى اسم حسابك على GitHub.

In [ ]:
USER = 'YOUR_GITHUB_USER'
!git clone https://github.com/$USER/maliki-rag.git /content/maliki-rag
%cd /content/maliki-rag
!pip install -q -r requirements.txt && pip install -q -e .

## 2) النصوص الخام
ضع ملفات الكتب في `MyDrive/maliki-rag/raw/` بالأسماء الواردة في `data/books.csv`، ثم انسخها:

In [ ]:
!cp /content/drive/MyDrive/maliki-rag/raw/* data/raw/ && ls data/raw

## 3) الاستيعاب والتقطيع وربط المتن بالشرح

In [ ]:
!python -m maliki_rag.cli ingest
# راجع عينة من المقاطع بعينك قبل المتابعة
import json, itertools
for l in itertools.islice(open('data/processed/chunks.jsonl', encoding='utf-8'), 3, 6):
    c = json.loads(l); print(c['id'], c['path'], c['matn_ref'], '\n', c['text'][:300], '\n---')

## 4) بناء الفهرس (مرة واحدة، ويُحفظ على Drive)

In [ ]:
!python -m maliki_rag.cli index

## 5) جرّب السؤال

In [ ]:
!python -m maliki_rag.cli ask "ما حكم وضع حزام في لباس إحرام الرجل؟"

## 6) التقييم بعد كل تعديل
املأ `eval/gold_questions.jsonl` أولاً.

In [ ]:
!python -m maliki_rag.cli eval --full

## 7) حفظ تعديلاتك في GitHub
من القائمة: **File → Save a copy in GitHub**، أو عبر `git commit` بعد إعداد رمز وصول شخصي.